# Experimento: Modelado Matemático de Pozos Reales (Vaca Muerta)

Este notebook toma los datos limpios del preprocesador y pone a prueba el modelo matemático `ArpsModeler` construido en `src/models/arps.py`.
Vamos a:
1. Cargar la data procesada (Parquet).
2. Identificar algunos de los pozos petroleros más productivos de **YPF S.A.**.
3. Ajustar el modelo empírico (Modified Arps) usando regresión no lineal.
4. Visualizar la curva histórica vs. la proyección a futuro y calcular el EUR.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Configurar ruta para poder importar src/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.arps import ArpsModeler

# Estilo de gráficos más profesional
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 1. Cargar datos
data_path = project_root / 'data' / 'processed' / 'vaca_muerta_prod.parquet'
df = pd.read_parquet(data_path)
print(f"Total de registros: {len(df):,}")

In [ ]:
# 2. Filtrar pozos de YPF con más de 24 meses de historia para tener un buen ajuste
df_ypf = df[df['empresa'].str.contains('YPF', na=False, case=False)].copy()

# Buscar los pozos con mayor producción de petróleo (pet_acumulado máximo)
pozos_top = df_ypf.groupby(['idpozo', 'sigla'])['pet_acumulado'].max().reset_index()
pozos_top = pozos_top.sort_values(by='pet_acumulado', ascending=False)

# Filtrar para que tengan suficientes datos de tiempo
conteos = df_ypf['idpozo'].value_counts()
pozos_validos = conteos[conteos > 24].index
pozos_top = pozos_top[pozos_top['idpozo'].isin(pozos_validos)]

top_4_ids = pozos_top.head(4)['idpozo'].tolist()
top_4_siglas = pozos_top.head(4)['sigla'].tolist()

print("Top 4 pozos seleccionados:", top_4_siglas)

In [ ]:
# 3. Ajuste y Visualización
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (pozo_id, sigla) in enumerate(zip(top_4_ids, top_4_siglas)):
    ax = axes[i]
    
    # Extraer data del pozo
    df_pozo = df_ypf[df_ypf['idpozo'] == pozo_id].sort_values('mes_relativo')
    
    # Limpiar datos anómalos: los pozos de shale suelen alcanzar su pico en el mes 2 o 3.
    # A veces el mes 1 es solo limpieza. Tomamos a partir del pico para el declino.
    meses = df_pozo['mes_relativo'].values
    produccion = df_pozo['prod_pet'].values
    
    if len(produccion) < 5:
        continue
        
    idx_pico = np.argmax(produccion)
    
    # Datos para ajustar (desde el pico en adelante)
    t_fit = meses[idx_pico:]
    q_fit = produccion[idx_pico:]
    
    # Evitar problemas si hay paros (producción = 0) en el medio
    mask = q_fit > 0
    t_fit = t_fit[mask] - t_fit[mask][0]  # Normalizar t para que arranque en 0 en el pico
    q_fit = q_fit[mask]
    
    # Instanciar el modelo Modificado (switch automático a exponencial)
    modeler = ArpsModeler(model_type='modified', d_term_annual=0.08)
    
    try:
        params = modeler.fit(t_fit, q_fit)
        
        # Predicción a largo plazo (ej. 120 meses / 10 años)
        t_pred = np.arange(0, 120)
        q_pred = modeler.predict(t_pred)
        
        # Calcular EUR (Estimated Ultimate Recovery a 10 años)
        eur = modeler.calculate_eur(120)
        
        # Graficar historia original vs ajuste
        t_plot_hist = meses[idx_pico:][mask]
        ax.scatter(t_plot_hist, q_fit, color='black', s=20, alpha=0.6, label='Producción Real')
        
        # Graficar predicción (sumamos el offset del pico para alinear el eje x)
        t_plot_pred = t_pred + meses[idx_pico]
        ax.plot(t_plot_pred, q_pred, color='red', linewidth=2, 
                label=f'Modified Arps (b={params["b"]:.2f})')
        
        ax.set_title(f"Pozo: {sigla}\nEUR (10 años): {eur:,.0f} m³", fontsize=12, fontweight='bold')
        ax.set_xlabel("Meses de Producción")
        ax.set_ylabel("Producción de Petróleo (m³/mes)")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.7)
        
    except Exception as e:
        ax.set_title(f"Error al ajustar pozo {sigla}")
        print(f"Error en pozo {sigla}: {e}")

plt.tight_layout()
plt.show()